[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/differentiation_course.ipynb)

# Differentiation full course: practice notebook

One practice question for every teaching chapter of the video, in the same order, and graphs you can move. Work each
question out on paper first, then type your answer into the box and press **check**. It tells you whether you're
right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

**Typing answers.** Write a derivative in x, with `*` or a space for times and `^` for powers: `2x + 3`,
`3x^2 cos(x^3)`, `e^(2x)`, `ln(x)`, `sqrt(x)`, `sec(x)^2`. Inverse functions are `asin(x)`, `atan(x)`. Fractions as
`-4/5`, π as `pi`. For several answers, separate them with commas, in the order the question asks.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, cmath, re
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, FloatSlider, FloatLogSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the function · slopes · areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
x, t = sp.symbols("x t")
r = sp.Rational
NAMES = {"pi": sp.pi, "e": sp.E, "x": x, "t": t, "ln": sp.log, "sqrt": sp.sqrt, "asin": sp.asin, "acos": sp.acos,
         "atan": sp.atan, "arcsin": sp.asin, "arccos": sp.acos, "arctan": sp.atan, "sec": sp.sec, "cosec": sp.csc,
         "csc": sp.csc, "cot": sp.cot}

def read(s):
    """Your typed answer as exact expressions (a list, one for each answer)."""
    s = (s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("π", "pi").replace("²", "^2")
          .replace("³", "^3").replace("′", "'"))
    for fn in ("sin", "cos", "tan"):                                  # sin⁻¹x and sin^-1 x → asin x
        s = s.replace(fn + "⁻¹", "a" + fn).replace(fn + "^-1", "a" + fn).replace(fn + "^(-1)", "a" + fn)
    s = re.sub(r"\|([^|]+)\|", r"(Abs(\1))", s)                       # |x| → (Abs(x))
    s = re.sub(r"√\s*(\d+(?:\.\d+)?|pi|\w)", r"sqrt(\1)", s).replace("√", "sqrt")   # √3 → sqrt(3); √(…) stays
    return [parse_expr(p, local_dict=NAMES, transformations=RULES) for p in s.split(",") if p.strip()]

POINTS = [0.3, 0.7, 1.3, 1.9, 2.6, -0.4, -1.7]               # where two expressions in x are compared

def same(a, b):
    """Equal as numbers, or (for expressions in x or t) equal at every sample point where the right answer is a real
    number."""
    a, b = sp.sympify(a), sp.sympify(b)
    if (a.free_symbols | b.free_symbols) - {x, t}: return False        # a letter we don't know
    var = (a.free_symbols | b.free_symbols or {x}).pop()
    tried = 0
    for v in POINTS:
        try: wb = complex(sp.N(b.subs(var, v), 30))
        except (TypeError, ValueError): continue
        if not cmath.isfinite(wb) or abs(wb.imag) > 1e-12: continue    # the right answer isn't defined here
        try: wa = complex(sp.N(a.subs(var, v), 30))
        except (TypeError, ValueError): return False
        if not abs(wa - wb) < 1e-9 * max(1, abs(wb)): return False
        tried += 1
    return tried >= 3

def nice(e):
    """An expression the way you'd type it."""
    return (sp.sstr(e).replace("**", "^").replace("exp(", "e^(").replace("log(", "ln(").replace("Abs(", "abs(")
            .replace("*", "·"))

# Each question: the question, the exact answers in order, the working, and (if it has one) a picture the check draws
# with your answers on it.
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = all(same(g, w) for g, w in zip(got, want))
            print("✓ right" if ok else "✗ not quite: try again, or press show working")
            if "draw" in q: q["draw"](got)
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def derivative_of(expr):
    """The derivative, worked out by sympy: what every answer below is checked against."""
    return sp.diff(expr, x)

## 1 · Rates of change

A chord from x = a to x = a + h has slope (f(a + h) − f(a))/h: the average rate of change. Shrink h and the chord
turns into the tangent: its slope is the instantaneous rate of change. Working that limit out is called
differentiating **from first principles**. Here f(x) = x² + 3x and a = 2. Shrink the step h and watch the chord's
slope settle.

In [ ]:
f1 = lambda v: v * v + 3 * v

def chord(h=1.0):
    xs = np.linspace(-0.5, 4.5, 300)
    fig, ax = plt.subplots(figsize=(8.6, 5))
    ax.plot(xs, f1(xs), color=TEAL, label="f(x) = x² + 3x")
    s = (f1(2 + h) - f1(2)) / h
    ax.plot(xs, f1(2) + s * (xs - 2), color=CORAL, lw=2, label=f"chord: slope {s:.4f}")
    ax.plot(xs, f1(2) + 7 * (xs - 2), color=SAGE, lw=1.4, ls="--", label="tangent at 2: slope 7")
    ax.plot([2, 2 + h], [f1(2), f1(2 + h)], "o", color=CHALK, ms=9, zorder=5)
    ax.set_xlim(-0.5, 4.5); ax.set_ylim(-5, 32); ax.set_xlabel("x"); ax.legend(loc="upper left")
    ax.set_title(f"h = {h:.4g}:  (f(2 + h) − f(2)) / h = {s:.4f}")
    plt.show()

assert all(abs((f1(2 + h) - f1(2)) / h - (7 + h)) < 1e-9 for h in (1, 0.1, 0.01, 0.001))   # the chord slope is 7 + h

In [ ]:
interact(chord, h=FloatLogSlider(value=1, base=10, min=-3, max=0.2, step=0.1, description="h"));

In [ ]:
Q[1] = {"text": "Differentiate f(x) = x² + 3x from first principles. Give f′(x), then the slope of the curve at x = 2.",
        "answer": [2 * x + 3, 7],
        "working": ["f(x + h) = x² + 2xh + h² + 3x + 3h.",
                    "f(x + h) − f(x) = 2xh + h² + 3h.",
                    "Divide by h: 2x + h + 3. Let h go to 0: f′(x) = 2x + 3.",
                    "At x = 2: 2 × 2 + 3 = 7, the number the chords settle on above."]}
assert derivative_of(x ** 2 + 3 * x) == 2 * x + 3
practice(1)

## 2 · Limits

A limit is where a function is heading as x gets close to a number, never what it does at the number. 0/0 means a
factor is hiding: factorise and cancel it.

In [ ]:
Q[2] = {"text": "Find the limit as x → 3 of (x² − 2x − 3)/(x − 3).",
        "answer": [4],
        "working": ["At 3 it's 0/0, so a factor is hiding: x² − 2x − 3 = (x − 3)(x + 1).",
                    "Cancel x − 3, leaving x + 1, which heads for 3 + 1 = 4."]}
assert sp.limit((x ** 2 - 2 * x - 3) / (x - 3), x, 3) == 4
practice(2)

## 3 · The derivative

f′(x) = the limit, as h → 0, of (f(x + h) − f(x))/h. It's a new function: the slope of f at every x. Drag x along
the curve f(x) = x³ − 3x: the tangent follows the slider, and the lower graph plots its slope, 3x² − 3.

In [ ]:
f3 = lambda v: v ** 3 - 3 * v
d3 = lambda v: 3 * v * v - 3

def tangent(x0=-1.5):
    xs = np.linspace(-2.3, 2.3, 300)
    fig, (top, low) = plt.subplots(2, 1, figsize=(8.6, 6.6), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
    top.plot(xs, f3(xs), color=TEAL, label="f(x) = x³ − 3x")
    near_ = np.linspace(x0 - 0.9, x0 + 0.9, 2)
    top.plot(near_, f3(x0) + d3(x0) * (near_ - x0), color=CORAL, lw=3, label="the tangent")
    top.plot([x0], [f3(x0)], "o", color=CHALK, ms=9, zorder=5)
    top.set_ylim(-4, 4); top.legend(loc="upper left")
    top.set_title(f"x = {x0:+.1f}:  slope of the tangent = {d3(x0):+.2f}")
    low.plot(xs, d3(xs), color=SAGE, label="f′(x) = 3x² − 3")
    low.plot([x0], [d3(x0)], "o", color=CORAL, ms=9, zorder=5)
    low.axhline(0, color=DIM, lw=1); low.set_ylim(-4, 13); low.set_xlabel("x"); low.legend(loc="upper center")
    plt.show()

assert derivative_of(x ** 3 - 3 * x) == 3 * x ** 2 - 3

In [ ]:
interact(tangent, x0=FloatSlider(value=-1.5, min=-2, max=2, step=0.1, description="x"));

In [ ]:
Q[3] = {"text": "Differentiate f(x) = 1/(x + 2) from first principles. Give f′(x), then f′(0).",
        "answer": [-1 / (x + 2) ** 2, r(-1, 4)],
        "working": ["f(x + h) − f(x) = 1/(x + h + 2) − 1/(x + 2) = −h / ((x + h + 2)(x + 2)).",
                    "Divide by h: −1 / ((x + h + 2)(x + 2)).",
                    "Let h go to 0: f′(x) = −1/(x + 2)².",
                    "f′(0) = −1/4."]}
assert sp.simplify(derivative_of(1 / (x + 2)) + 1 / (x + 2) ** 2) == 0
practice(3)

## 4 · Continuity and differentiability

Differentiable means continuous **and** with no corner: at a join, the values must meet and the slopes must match.

In [ ]:
Q[4] = {"text": "f(x) = x² when x ≤ 1, and f(x) = ax + b when x > 1. Find a and b so that f is differentiable at 1. "
                "Give a, then b.",
        "answer": [2, -1],
        "working": ["Slopes match at 1: the left piece has slope 2x = 2, the right piece has slope a. So a = 2.",
                    "Values meet at 1: 1² = a + b, so 1 = 2 + b and b = −1."]}
assert 1 ** 2 == 2 * 1 - 1 and sp.diff(x ** 2, x).subs(x, 1) == 2
practice(4)

## 5 · The power rule and the basic rules

d/dx xⁿ = nxⁿ⁻¹, for any power n. Constants multiply through, and sums are differentiated term by term.

In [ ]:
Q[5] = {"text": "Differentiate 5x⁴ − 3/x + 4√x.",
        "answer": [20 * x ** 3 + 3 / x ** 2 + 2 / sp.sqrt(x)],
        "working": ["Write it with powers: 5x⁴ − 3x⁻¹ + 4x^(1/2).",
                    "Power rule, term by term: 20x³ + 3x⁻² + 2x^(−1/2).",
                    "That's 20x³ + 3/x² + 2/√x."]}
assert sp.simplify(derivative_of(5 * x ** 4 - 3 / x + 4 * sp.sqrt(x)) - Q[5]["answer"][0]) == 0
practice(5)

## 6 · The product and quotient rules

(uv)′ = uv′ + vu′, and (u/v)′ = (vu′ − uv′)/v².

In [ ]:
Q[6] = {"text": "Differentiate x² sin x, then (x + 1)/(x − 1). Give the two derivatives, in that order.",
        "answer": [2 * x * sp.sin(x) + x ** 2 * sp.cos(x), -2 / (x - 1) ** 2],
        "working": ["Product: u = x², v = sin x. uv′ + vu′ = x² cos x + 2x sin x.",
                    "Quotient: u = x + 1, v = x − 1. (vu′ − uv′)/v² = ((x − 1) − (x + 1))/(x − 1)² = −2/(x − 1)²."]}
assert sp.simplify(derivative_of(x ** 2 * sp.sin(x)) - Q[6]["answer"][0]) == 0
assert sp.simplify(derivative_of((x + 1) / (x - 1)) - Q[6]["answer"][1]) == 0
practice(6)

## 7 · Derivatives of trigonometric functions

(sin x)′ = cos x, (cos x)′ = −sin x, (tan x)′ = sec²x, (sec x)′ = sec x tan x, (cosec x)′ = −cosec x cot x,
(cot x)′ = −cosec²x. All with x in radians.

In [ ]:
Q[7] = {"text": "Differentiate x tan x.",
        "answer": [sp.tan(x) + x * sp.sec(x) ** 2],
        "working": ["Product rule with u = x and v = tan x.",
                    "uv′ + vu′ = x sec²x + tan x × 1 = tan x + x sec²x."]}
assert sp.simplify(derivative_of(x * sp.tan(x)) - Q[7]["answer"][0]) == 0
practice(7)

## 8 · The chain rule

For a function inside a function: differentiate the outside, keep the inside as it is, then multiply by the
derivative of the inside.

In [ ]:
Q[8] = {"text": "Differentiate (2x + 3)⁵, then cos(x³). Give the two derivatives, in that order.",
        "answer": [10 * (2 * x + 3) ** 4, -3 * x ** 2 * sp.sin(x ** 3)],
        "working": ["(2x + 3)⁵: outside u⁵ gives 5u⁴; inside 2x + 3 gives 2. So 5(2x + 3)⁴ × 2 = 10(2x + 3)⁴.",
                    "cos(x³): outside gives −sin(x³); inside gives 3x². So −3x² sin(x³)."]}
assert sp.simplify(derivative_of((2 * x + 3) ** 5) - Q[8]["answer"][0]) == 0
assert sp.simplify(derivative_of(sp.cos(x ** 3)) - Q[8]["answer"][1]) == 0
practice(8)

## 9 · Exponential and logarithmic functions

(eˣ)′ = eˣ, (aˣ)′ = aˣ ln a, (ln x)′ = 1/x, and (log base a of x)′ = 1/(x ln a).

In [ ]:
Q[9] = {"text": "Differentiate e^(x²), then ln(3x + 1). Give the two derivatives, in that order.",
        "answer": [2 * x * sp.exp(x ** 2), 3 / (3 * x + 1)],
        "working": ["e^(x²): the chain rule gives e^(x²) × 2x = 2x e^(x²).",
                    "ln(3x + 1): the chain rule gives 1/(3x + 1) × 3 = 3/(3x + 1)."]}
assert sp.simplify(derivative_of(sp.exp(x ** 2)) - Q[9]["answer"][0]) == 0
assert sp.simplify(derivative_of(sp.log(3 * x + 1)) - Q[9]["answer"][1]) == 0
practice(9)

## 10 · Implicit differentiation

Differentiate both sides with respect to x, treating y as a function of x: so y² gives 2y dy/dx. Then solve for
dy/dx.

In [ ]:
Q[10] = {"text": "The curve x² + xy + y² = 7 passes through (1, 2). Find dy/dx there.",
         "answer": [r(-4, 5)],
         "working": ["Check the point: 1 + 2 + 4 = 7.",
                     "Differentiate: 2x + (y + x dy/dx) + 2y dy/dx = 0.",
                     "So dy/dx = −(2x + y)/(x + 2y).",
                     "At (1, 2): −(2 + 2)/(1 + 4) = −4/5."]}
_y = sp.Function("y")(x)
_dy = sp.solve(sp.diff(x ** 2 + x * _y + _y ** 2, x), sp.diff(_y, x))[0]
assert 1 + 1 * 2 + 2 ** 2 == 7 and _dy.subs(_y, 2).subs(x, 1) == r(-4, 5)
practice(10)

## 11 · Inverse trigonometric functions

(sin⁻¹x)′ = 1/√(1 − x²), (cos⁻¹x)′ = −1/√(1 − x²), (tan⁻¹x)′ = 1/(1 + x²). Type them as `asin`, `acos`, `atan`.

In [ ]:
Q[11] = {"text": "Differentiate tan⁻¹(3x), then sin⁻¹(x/2). Give the two derivatives, in that order.",
         "answer": [3 / (1 + 9 * x ** 2), 1 / sp.sqrt(4 - x ** 2)],
         "working": ["tan⁻¹(3x): 1/(1 + (3x)²) × 3 = 3/(1 + 9x²).",
                     "sin⁻¹(x/2): 1/√(1 − x²/4) × 1/2 = 1/(2√((4 − x²)/4)) = 1/√(4 − x²)."]}
assert sp.simplify(derivative_of(sp.atan(3 * x)) - Q[11]["answer"][0]) == 0
assert same(derivative_of(sp.asin(x / 2)), Q[11]["answer"][1])
practice(11)

## 12 · Logarithmic differentiation

For a variable power, or a long product, take ln of both sides first, then differentiate: (ln y)′ = y′/y.

In [ ]:
Q[12] = {"text": "y = x^(2x), for x > 0. Find dy/dx, then its value at x = 1.",
         "answer": [x ** (2 * x) * (2 * sp.log(x) + 2), 2],
         "working": ["ln y = 2x ln x.",
                     "Differentiate: y′/y = 2 ln x + 2x × 1/x = 2 ln x + 2.",
                     "So y′ = x^(2x) (2 ln x + 2).",
                     "At x = 1: 1 × (0 + 2) = 2."]}
assert same(derivative_of(x ** (2 * x)), Q[12]["answer"][0])
practice(12)

## 13 · Parametric differentiation

When x and y both depend on t: dy/dx = (dy/dt) ÷ (dx/dt).

In [ ]:
Q[13] = {"text": "x = t² and y = t³ − 3t. Find dy/dx when t = 2.",
         "answer": [r(9, 4)],
         "working": ["dx/dt = 2t and dy/dt = 3t² − 3.",
                     "dy/dx = (3t² − 3)/(2t).",
                     "At t = 2: (12 − 3)/4 = 9/4."]}
assert ((sp.diff(t ** 3 - 3 * t, t) / sp.diff(t ** 2, t)).subs(t, 2)) == r(9, 4)
practice(13)

## 14 · Higher derivatives

Differentiate again: f″ is the derivative of f′, and it measures how the slope itself changes.

In [ ]:
Q[14] = {"text": "y = x⁴ − 3x² + 2. Find the second derivative, then its value at x = 1.",
         "answer": [12 * x ** 2 - 6, 6],
         "working": ["y′ = 4x³ − 6x.", "y″ = 12x² − 6.", "At x = 1: 12 − 6 = 6."]}
assert sp.expand(sp.diff(x ** 4 - 3 * x ** 2 + 2, x, 2)) == 12 * x ** 2 - 6
practice(14)

## 15 · Tangents and normals

The tangent at (a, f(a)) has slope f′(a): y − f(a) = f′(a)(x − a). The normal is at right angles to it, with slope
−1/f′(a).

In [ ]:
Q[15] = {"text": "For y = x² − 4x + 5 at x = 3, give the slope of the tangent, the tangent's y-intercept, and the "
                 "slope of the normal, in that order.",
         "answer": [2, -4, r(-1, 2)],
         "working": ["At x = 3, y = 9 − 12 + 5 = 2.",
                     "y′ = 2x − 4, so the slope is 2.",
                     "Tangent: y − 2 = 2(x − 3), so y = 2x − 4: the y-intercept is −4.",
                     "Normal slope: −1/2."]}
assert (x ** 2 - 4 * x + 5).subs(x, 3) == 2 and sp.diff(x ** 2 - 4 * x + 5, x).subs(x, 3) == 2 and 2 - 2 * 3 == -4
practice(15)

## 16 · Rates of change

When two quantities are linked, their rates are linked too: differentiate the link with respect to time.

In [ ]:
Q[16] = {"text": "A circle's radius grows at 3 cm a second. How fast is its area growing when the radius is 5 cm, "
                 "in square centimetres a second? (Give it exactly, with pi.)",
         "answer": [30 * sp.pi],
         "working": ["A = πr², so dA/dt = 2πr × dr/dt.",
                     "dA/dt = 2π × 5 × 3 = 30π, about 94.2 cm² a second."]}
assert sp.diff(sp.pi * x ** 2, x).subs(x, 5) * 3 == 30 * sp.pi
practice(16)

## 17 · Linear approximation

Near a point you know, a curve is close to its tangent: f(a + Δx) ≈ f(a) + f′(a) Δx.

In [ ]:
Q[17] = {"text": "Use the tangent to y = √x at x = 64 to estimate √65. Give the estimate as a fraction or a decimal.",
         "answer": [r(129, 16)],
         "working": ["f(64) = 8, and f′(x) = 1/(2√x), so f′(64) = 1/16.",
                     "√65 ≈ 8 + 1/16 × 1 = 129/16 = 8.0625.",
                     "The true value is 8.06226…: the estimate is out by less than 0.0003."]}
assert abs(8 + 1 / 16 - math.sqrt(65)) < 3e-4
practice(17)

## 18 · Rolle's theorem and the mean value theorem

For f continuous on [a, b] and differentiable inside, somewhere in between the slope equals the average slope:
f′(c) = (f(b) − f(a))/(b − a).

In [ ]:
Q[18] = {"text": "For f(x) = x³ on [0, 3], find the c between 0 and 3 that the mean value theorem promises.",
         "answer": [sp.sqrt(3)],
         "working": ["Average slope: (27 − 0)/(3 − 0) = 9.",
                     "f′(c) = 3c² = 9, so c² = 3 and c = √3 (about 1.732), which is between 0 and 3."]}
assert 0 < math.sqrt(3) < 3 and abs(3 * math.sqrt(3) ** 2 - 9) < 1e-12
practice(18)

## 19 · Increasing and decreasing functions

Where f′ > 0, f is increasing; where f′ < 0, it is decreasing. Factor f′ and test the sign between its zeros.

In [ ]:
Q[19] = {"text": "f(x) = x³ − 3x² − 9x + 1 is decreasing on one interval. Give its two ends, smaller first.",
         "answer": [-1, 3],
         "working": ["f′(x) = 3x² − 6x − 9 = 3(x + 1)(x − 3).",
                     "Between −1 and 3, one factor is negative and one positive, so f′ < 0 there.",
                     "f is decreasing on (−1, 3), and increasing outside it."]}
assert sp.expand(sp.diff(x ** 3 - 3 * x ** 2 - 9 * x + 1, x) - 3 * (x - 3) * (x + 1)) == 0
practice(19)

## 20 · Maxima and minima

Critical points are where f′ = 0 (or doesn't exist). The first derivative test: + to − is a maximum, − to + a
minimum. On a closed interval, the absolute maximum and minimum are among the critical points **and** the two ends.
The check draws the curve with your answers marked on it.

In [ ]:
f20 = lambda v: 2 * v ** 3 - 9 * v ** 2 + 12 * v - 4
assert sp.expand(sp.diff(f20(x), x) - 6 * (x - 2) * (x - 1)) == 0
assert [f20(v) for v in (0, 1, 2, 3)] == [-4, 1, 0, 5]
assert max(f20(v) for v in np.linspace(0, 3, 3001)) <= 5 and min(f20(v) for v in np.linspace(0, 3, 3001)) >= -4

def draw20(got):
    vals = []
    for g in got:
        try: vals.append(float(g))
        except (TypeError, ValueError): return
    xs = np.linspace(-0.2, 3.2, 300)
    fig, ax = plt.subplots(figsize=(8.6, 4.0))
    ax.plot(xs, f20(xs), color=TEAL, label="f(x) = 2x³ − 9x² + 12x − 4")
    ax.axvspan(-0.2, 0, color=DIM, alpha=0.15, lw=0); ax.axvspan(3, 3.2, color=DIM, alpha=0.15, lw=0)
    for v, name in zip(vals[:2], ("your local max", "your local min")):
        ax.plot([v], [f20(v)], "o", color=CORAL, ms=10, zorder=5)
        right = v > 2.2                                               # near the right edge: label on the left
        ax.annotate(name, (v, f20(v)), xytext=(-10 if right else 8, 10 if "max" in name else -20),
                    textcoords="offset points", color=CORAL, ha="right" if right else "left")
    for v, name in zip(vals[2:], ("your absolute max", "your absolute min")):
        ax.axhline(v, color=CORAL, lw=1.2, ls="--")
        ax.text(-0.15, v + 0.25, name, color=CORAL, fontsize=12)
    ax.set_xlim(-0.2, 3.2); ax.set_ylim(-6, 7); ax.set_xlabel("x"); ax.legend(loc="lower right")
    ax.set_title("the curve on [0, 3], with your answers marked")
    plt.show()

In [ ]:
Q[20] = {"text": "f(x) = 2x³ − 9x² + 12x − 4 on the interval [0, 3]. Give the x of the local maximum, the x of the "
                 "local minimum, the absolute maximum value and the absolute minimum value, in that order.",
         "answer": [1, 2, 5, -4],
         "working": ["f′(x) = 6x² − 18x + 12 = 6(x − 1)(x − 2): critical points at 1 and 2.",
                     "f′ goes + to − at 1 (a local maximum, f(1) = 1) and − to + at 2 (a local minimum, f(2) = 0).",
                     "The ends: f(0) = −4 and f(3) = 54 − 81 + 36 − 4 = 5.",
                     "Compare −4, 1, 0 and 5: the absolute maximum is 5 (at x = 3), the minimum −4 (at x = 0)."],
         "draw": draw20}
practice(20)

## 21 · Concavity and the second derivative test

f″ > 0: concave up (a cup); f″ < 0: concave down. A point of inflection is where the concavity changes. At a
critical point, f″ > 0 means a minimum and f″ < 0 a maximum.

In [ ]:
Q[21] = {"text": "f(x) = x³ − 6x² + 5. Give the x of the point of inflection, then the x of the local maximum and "
                 "the x of the local minimum (use the second derivative test).",
         "answer": [2, 0, 4],
         "working": ["f′(x) = 3x² − 12x = 3x(x − 4): critical points at 0 and 4.",
                     "f″(x) = 6x − 12, which is zero at 2 and changes sign there: the inflection is at x = 2.",
                     "f″(0) = −12 < 0: a maximum at 0. f″(4) = 12 > 0: a minimum at 4."]}
_f21 = x ** 3 - 6 * x ** 2 + 5
assert sp.solve(sp.diff(_f21, x, 2), x) == [2] and sp.solve(sp.diff(_f21, x), x) == [0, 4]
assert sp.diff(_f21, x, 2).subs(x, 0) == -12 and sp.diff(_f21, x, 2).subs(x, 4) == 12
practice(21)

## 22 · Optimisation

Write the quantity as a function of one variable, differentiate, set the derivative to zero, check it's the
maximum or minimum you want, and check the ends.

In [ ]:
Q[22] = {"text": "A 12 cm by 12 cm square of card has a square of side x cut from each corner, and the sides folded "
                 "up into an open box. Give the x that makes the volume largest, then the largest volume in cm³.",
         "answer": [2, 128],
         "working": ["V = x(12 − 2x)², for x between 0 and 6.",
                     "V′ = (12 − 2x)² − 4x(12 − 2x) = (12 − 2x)(12 − 6x).",
                     "V′ = 0 at x = 2 (x = 6 makes no box). V(2) = 2 × 8² = 128 cm³.",
                     "The ends give V = 0, so 128 is the largest."]}
_V = x * (12 - 2 * x) ** 2
assert sp.solve(sp.diff(_V, x), x) == [2, 6] and _V.subs(x, 2) == 128
assert max(float(_V.subs(x, v)) for v in np.linspace(0, 6, 601)) <= 128
practice(22)

## Sources
NCERT Mathematics Class XI, ch. 12 (Limits and Derivatives); Class XII, ch. 5 (Continuity and Differentiability) and
ch. 6 (Application of Derivatives); OpenStax, Calculus Volume 1, ch. 3–4. All retrieved 2026-10-31. Every question
is our own.